In [1]:
import numpy as np

X_train = np.load("../dataset/processed/X_train.npy")
X_test = np.load("../dataset/processed/X_test.npy")

y_train = np.load("../dataset/processed/y_train.npy")
y_test = np.load("../dataset/processed/y_test.npy")

print(X_train.shape)
print(X_test.shape)

(1632, 224, 224, 3)
(409, 224, 224, 3)


In [2]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

In [3]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
base_model.trainable = False

In [5]:
from tensorflow.keras.models import Sequential

model = Sequential([
    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 56s 901ms/step - accuracy: 0.5379 - loss: 0.7835 - val_accuracy: 0.6055 - val_loss: 0.6681
Epoch 2/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 35s 863ms/step - accuracy: 0.6352 - loss: 0.6371 - val_accuracy: 0.6483 - val_loss: 0.6570
Epoch 3/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 32s 783ms/step - accuracy: 0.6483 - loss: 0.6321 - val_accuracy: 0.6361 - val_loss: 0.6420
Epoch 4/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 34s 824ms/step - accuracy: 0.6682 - loss: 0.6085 - val_accuracy: 0.6208 - val_loss: 0.6460
Epoch 5/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 43s 878ms/step - accuracy: 0.6904 - loss: 0.5814 - val_accuracy: 0.6116 - val_loss: 0.6594
Epoch 6/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 37s 778ms/step - accuracy: 0.6958 - loss: 0.5760 - val_accuracy: 0.6514 - val_loss: 0.6514
Epoch 7/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 31s 767ms/step - accuracy: 0.7111 - loss: 0.5442 - val_accuracy: 0.6208 - val_loss: 0.6495
Epoch 8/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 32s 772ms/step - accuracy: 0.7142 - loss: 0.5396 - val_accu

In [9]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", test_accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 576ms/step - accuracy: 0.6015 - loss: 0.7034
Test Accuracy: 0.6014670133590698


In [10]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)

print(cm)

13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 963ms/step
[[125  92]
 [ 71 121]]


In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.58      0.61       217
           1       0.57      0.63      0.60       192

    accuracy                           0.60       409
   macro avg       0.60      0.60      0.60       409
weighted avg       0.61      0.60      0.60       409



In [12]:
model.save("../model/deepshield_mobilenetv2.keras")

print("Model Saved!")

Model Saved!


In [13]:
base_model.trainable = False

In [14]:
base_model.trainable = True

In [15]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [16]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
history_finetune = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 60s 924ms/step - accuracy: 0.6345 - loss: 0.7341 - val_accuracy: 0.6361 - val_loss: 0.6592
Epoch 2/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 34s 820ms/step - accuracy: 0.6705 - loss: 0.6748 - val_accuracy: 0.6361 - val_loss: 0.6568
Epoch 3/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 35s 844ms/step - accuracy: 0.6812 - loss: 0.6230 - val_accuracy: 0.6391 - val_loss: 0.6561
Epoch 4/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 41s 832ms/step - accuracy: 0.7203 - loss: 0.5604 - val_accuracy: 0.6483 - val_loss: 0.6550
Epoch 5/5
41/41 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.7441 - loss: 0.5313 - val_accuracy: 0.6483 - val_loss: 0.6576


In [18]:
model.save("../model/deepshield_mobilenetv2.keras")